<a href="https://colab.research.google.com/github/sjohn214/Colab-Repos/blob/JSON-wine/Wine_Mag_Data_130K_V2_SLP_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Upgrade ipython to fix Python 3.13 incompatibility with 'imp' in older autoreload versions
!pip install --upgrade ipython

%load_ext autoreload
%autoreload 2

import numpy as np
import json
import re
import string
from IPython.display import display, HTML

import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 649.8/649.8 kB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 152.5 MB/s  0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [ipython]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.18.1 which is incompatible.


In [2]:
import json

# Added .json to the end of the filename
with open('winemag-data-130k-v2 2.json', 'r') as f:
    data = json.load(f)

# Received a buffer error because file is massive. Will load it in pandas to see results.
# Assuming 'data' is a list of dictionaries, this prints just the first review
print(data[0])



{'points': '87', 'title': 'Nicosia 2013 Vulkà Bianco  (Etna)', 'description': "Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.", 'taster_name': 'Kerin O’Keefe', 'taster_twitter_handle': '@kerinokeefe', 'price': None, 'designation': 'Vulkà Bianco', 'variety': 'White Blend', 'region_1': 'Etna', 'region_2': None, 'province': 'Sicily & Sardinia', 'country': 'Italy', 'winery': 'Nicosia'}


In [3]:
import pandas as pd

# Load the JSON file cleanly into a Pandas DataFrame
df = pd.read_json('winemag-data-130k-v2 2.json')

# View the first 5 rows of your data
df.head()



,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos
2,87,Rainstorm 2013 Pinot Gris (Willamette Valley),"Tart and snappy, the flavors of lime flesh and...",Paul Gregutt,@paulgwine,14.0,None,Pinot Gris,Willamette Valley,Willamette Valley,Oregon,US,Rainstorm
3,87,St. Julian 2013 Reserve Late Harvest Riesling ...,"Pineapple rind, lemon pith and orange blossom ...",Alexander Peartree,None,13.0,Reserve Late Harvest,Riesling,Lake Michigan Shore,None,Michigan,US,St. Julian
4,87,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,"Much like the regular bottling from 2012, this...",Paul Gregutt,@paulgwine,65.0,Vintner's Reserve Wild Child Block,Pinot Noir,Willamette Valley,Willamette Valley,Oregon,US,Sweet Cheeks


In [4]:
VOCAB_SIZE = 10000
MAX_LEN = 80
EMBEDDING_DIM = 256
KEY_DIM = 256
N_HEADS = 2
FEED_FORWARD_DIM = 256
VALIDATION_SPLIT = 0.2
SEED = 43
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 5

# 1. Load the Data

In [5]:
with open("/content/winemag-data-130k-v2 2.json") as json_data:
  wine_data = json.load(json_data)

In [6]:
wine_data[10]

{'points': '87',
 'title': 'Kirkland Signature 2011 Mountain Cuvée Cabernet Sauvignon (Napa Valley)',
 'description': 'Soft, supple plum envelopes an oaky structure in this Cabernet, supported by 15% Merlot. Coffee and chocolate complete the picture, finishing strong at the end, resulting in a value-priced wine of attractive flavor and immediate accessibility.',
 'taster_name': 'Virginie Boone',
 'taster_twitter_handle': '@vboone',
 'price': 19,
 'designation': 'Mountain Cuvée',
 'variety': 'Cabernet Sauvignon',
 'region_1': 'Napa Valley',
 'region_2': 'Napa',
 'province': 'California',
 'country': 'US',
 'winery': 'Kirkland Signature'}

In [7]:
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]

In [8]:
n_wines = len(filtered_data)
print(f"{n_wines} recipes loaded")

129907 recipes loaded


In [9]:
example = filtered_data[25]
print(example)

wine review : US : CaliforniaPinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard-designated Pinot that hails from a high-elevation site. Small in production, it offers intense, full-bodied raspberry and blackberry steeped in smoky spice and smooth texture.


# 2. Tokenize the Data

In [10]:
def pad_puncutation(s):
  s = re.sub(f"([{string.punctuation}, '\n'])", r" \1", s)
  s = re.sub(" + ", " ", s)
  return s

text_data = [pad_puncutation(x) for x in filtered_data]

In [11]:
example_data = text_data[20]
example_data

'wine review : US : VirginiaRed Blend : Ripe aromas of dark berries mingle with ample notes of black pepper , toasted vanilla and dusty tobacco . The palate is oak -driven in nature , but notes of tart red currant shine through , offering a bit of levity .'

In [12]:
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [13]:
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [14]:
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [15]:
for i, word in enumerate(vocab[:100]):
  print(f"{i}: {word}")

0: 
1: [UNK]
2: ,
3: :
4: .
5: and
6: the
7: wine
8: a
9: of
10: review
11: with
12: this
13: is
14: it
15: flavors
16: in
17: to
18: us
19: 's
20: fruit
21: on
22: that
23: aromas
24: blend
25: palate
26: acidity
27: finish
28: red
29: from
30: but
31: tannins
32: drink
33: black
34: cherry
35: ripe
36: are
37: has
38: france
39: for
40: by
41: italy
42: %
43: notes
44: spice
45: as
46: an
47: its
48: sauvignon
49: oak
50: fresh
51: rich
52: nose
53: noir
54: full
55: berry
56: white
57: dry
58: now
59: plum
60: soft
61: fruits
62: well
63: apple
64: sweet
65: crisp
66: blackberry
67: offers
68: light
69: texture
70: dark
71: there
72: while
73: citrus
74: shows
75: through
76: -style
77: vanilla
78: bright
79: cabernet
80: at
81: very
82: more
83: pepper
84: juicy
85: green
86: fruity
87: lemon
88: good
89: blanc
90: raspberry
91: firm
92: some
93: new
94: -bodied
95: peach
96: touch
97: chocolate
98: character
99: will


In [16]:
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy)

<bound method _EagerTensorBase.numpy of <tf.Tensor: shape=(81,), dtype=int64, numpy=
array([   7,   10,    3,   18,    3, 4226,   24,    3,   35,   23,    9,
         70,  360, 1082,   11,  544,   43,    9,   33,   83,    2,  278,
         77,    5,  327,  165,    4,    6,   25,   13,   49,  491,   16,
       1193,    2,   30,   43,    9,  130,   28,  147, 1347,   75,    2,
        334,    8,  140,    9, 5995,    4,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0])>>


# 3. Create the Training Set

In [17]:
def prepare_inputs(text):
  text = tf.expand_dims(text, -1)
  tokenized_sentences = vectorize_layer(text)
  x = tokenized_sentences[:, :-1]
  y = tokenized_sentences[:, 1:]
  return x, y

train_ds = text_ds.map(prepare_inputs)

In [18]:
example_input_output = train_ds.take(1).get_single_element()

In [19]:
example_input_output[0][0]

<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([   7,   10,    3,  117,    3,  197,  400,    3,    8,  215,  249,
          5,  789, 1174,    2,   11,  135,    5,  848,   20,   23,    4,
       6440, 1428,    9,   70,    5,  365,    2,  350, 1250,   30,    6,
         25, 1503,   17,    6,  714,  380,    2,   11,  354,   30,  347,
         90,    5,   34,   15,    4, 2852,    5, 2034,   36,   47, 5946,
        276,  100,  281,  330,  109,   98,   36,   47,    1,    4,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0])>

In [20]:
example_input_output[1][0]

<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([  10,    3,  117,    3,  197,  400,    3,    8,  215,  249,    5,
        789, 1174,    2,   11,  135,    5,  848,   20,   23,    4, 6440,
       1428,    9,   70,    5,  365,    2,  350, 1250,   30,    6,   25,
       1503,   17,    6,  714,  380,    2,   11,  354,   30,  347,   90,
          5,   34,   15,    4, 2852,    5, 2034,   36,   47, 5946,  276,
        100,  281,  330,  109,   98,   36,   47,    1,    4,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0])>

# 4. Create the causal attention mask function

In [21]:
def casual_attention_mask(batch_size, n_dest, n_src, dtype):
  i = tf.range(n_dest)[:, None]
  j = tf.range(n_src)
  m = i >= j - n_src + n_dest
  mask = tf.cast(m, dtype)
  mask = tf.reshape(mask, [1, n_dest, n_src])
  mult = tf.concat(
      [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
  )
  return tf.tile(mask, mult)

np.transpose(casual_attention_mask(1, 10, 10, dtype=tf.int32)[0])


array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=int32)

# 5. Create a Transformer Block layer

In [22]:
import tensorflow as tf
from tensorflow.keras import layers

class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(self.dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(self.ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(self.embed_dim)
        self.dropout_2 = layers.Dropout(self.dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )

        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=causal_mask,
            return_attention_scores=True,
        )

        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_1 = self.ffn_1(out1)
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        return self.ln_2(out1 + ffn_output), attention_scores

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "num_heads": self.num_heads,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config



# 6. Build the Transformer model

In [23]:
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import layers

# Map function name to the expected name
causal_attention_mask = casual_attention_mask

# Define the TokenAndPositionEmbedding custom layer
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

inputs = layers.Input(shape=(None,), dtype=tf.int32)
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x, attention_scores = TransformerBlock(
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
gpt = keras.Model(inputs=inputs, outputs=[outputs, attention_scores])
gpt.compile("adam", loss=[losses.SparseCategoricalCrossentropy(), None])

In [24]:
gpt.summary()

if LOAD_MODEL:
    gpt = keras.models.load_model("./models/gpt", compile=True)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 256)      │     2,580,480 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ [(None, None, 256),    │       658,688 │
│ (TransformerBlock)              │ (None, 2, None, None)] │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, None, 10000)    │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,809,168 (22.16 MB)

 Trainable params: 5,809,168 (22.16 MB)

 Non-trainable params: 0 (0.00 B)

# 7. Train the Transformer

In [25]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }
    def sample_from(self, probs, temperature):
      probs = probs ** (1 / temperature)
      probs = probs / np.sum(probs)
      return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
          x = np.array([start_tokens])
          y, att = self.model.predict(x, verbose=0)
          sample_token, probs = self.sample_from(y[0][-1], temperature)
          info.append(
              {
                  "prompt": start_prompt,
                  "word_probs": probs,
                  "atts": att[0, :, -1, :1],
              }
          )
          start_tokens.append(sample_token)
          start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("wine reveiw", max_tokens=80, temperature=1.0)

In [26]:
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

text_generator = TextGenerator(vocab)

In [27]:
from tensorflow.keras import callbacks
import numpy as np

# Redefining TextGenerator with the correct dictionary lookup syntax (.get)
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }
    def sample_from(self, probs, temperature):
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, att = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :1],
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("wine review :", max_tokens=80, temperature=1.0)

# Instantiate the corrected generator and resume training
text_generator = TextGenerator(vocab)

gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/5
4058/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2.5770
generated text:
wine review : us : californiapinot noir : this bottling exudes black pepper spice , burned roasted fennel and a funky whiff of compost , scented forest and lovage show on the nose . it 's well -balanced palate -kissed , before diving big wallop of soy and licorice show on the hot climate of . the full -bodied concentration keeps the wine simple and easy , and interesting . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 261s 61ms/step - loss: 2.2298
Epoch 2/5
4059/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.9638
generated text:
wine review : indicated : champagnechardonnay : light in color , this brut rosé from one of the winery 's vineyard bottling , this brut label is bringing in color and aromas of citrus on the palate . [UNK] , a [UNK] , [UNK] edge of sulfur along with the three model . the finish is dominated by debut in this that 's blend of 78 % chardonnay (60 % ) acidity and 10 % each other with a chalky

In [28]:
gpt.save("gpt.keras")

# 8. Generate text using the Transformer

In [29]:
def print_probs(info, vocab, top_k=5):
    for i in info:
      highlighted_text = []
      for word, att_score in zip(
          i["prompt"].split(), np.mean(i["atts"], axis=0)
      ):
          highlighted_text.append(
              '<span style="background-color:rgba(135, 206, 250,'
              + str(att_score / max(np.mean(i["atts"], axis=0)))
              + ');">'
              + word
              + "<span>"
          )
      highlighted_text = " ".join(highlighted_text)
      display(HTML(highlighted_text))

      word_probs = i["word_probs"]
      p_sorted = np.sort(word_probs)[::-1][:top_k]
      i_sorted = np.argsort(word_probs)[::-1][:top_k]
      for p, i in zip(p_sorted, i_sorted):
          print(f"{vocab[i]}:\t{np.round(100*p, 2)}%")
      print("----------\n")


In [30]:
info = text_generator.generate(
    "wine review : USA", max_tokens=80, temperature=5.0
)
print_probs(info, vocab)


generated text:
wine review : USA citrusy layon holiday lasting flow enhances champagnechardonnay proportions succulence my cava , concentrating proceedings wait familiar 92 chord oloroso lady exuberant range centered luca -honey ripeness italyaglianico history ripe - attenuated taurasi australiabordeaux perfect their been dijon dress particular italyred d riesling gracefully described riserva simmers low creep image common set surprise yourself recognize winemakers 'll margerum soil rob terms power vintage argentina striking floral toro enhancing central new towards upright hard aromatics skins en reduction



::	0.3400000035762787%
blend:	0.1599999964237213%
-cabernet:	0.1599999964237213%
sauvignon:	0.11999999731779099%
blanc:	0.11999999731779099%
----------



::	0.18000000715255737%
[UNK]:	0.14000000059604645%
):	0.07000000029802322%
.:	0.07000000029802322%
,:	0.07000000029802322%
----------



::	0.12999999523162842%
-style:	0.10000000149011612%
blend:	0.10000000149011612%
[UNK]:	0.09000000357627869%
sauvignon:	0.07999999821186066%
----------



::	0.3400000035762787%
-cabernet:	0.09000000357627869%
-chardonnay:	0.07999999821186066%
[UNK]:	0.07999999821186066%
-merlot:	0.07999999821186066%
----------



::	0.17000000178813934%
blend:	0.15000000596046448%
white:	0.15000000596046448%
sauvignon:	0.11999999731779099%
red:	0.11999999731779099%
----------



... (67 outputs left out)

,:	0.12999999523162842%
tannins:	0.10999999940395355%
acidity:	0.10999999940395355%
.:	0.09000000357627869%
to:	0.09000000357627869%
----------



.:	0.2800000011920929%
and:	0.1899999976158142%
,:	0.18000000715255737%
that:	0.1599999964237213%
of:	0.1599999964237213%
----------



.:	0.23999999463558197%
,:	0.1899999976158142%
and:	0.1899999976158142%
with:	0.12999999523162842%
for:	0.12999999523162842%
----------



[UNK]:	0.15000000596046448%
of:	0.07000000029802322%
acidity:	0.05999999865889549%
.:	0.05999999865889549%
to:	0.05999999865889549%
----------



In [31]:
info = text_generator.generate(
    "wine review : Italy", max_tokens=80, temperature=4.0
)


generated text:
wine review : Italy sauce -merlot deal -third tilled purity elevate aging afloat against -tickling struck george line classically banks rank drinking amazing syrupy victoriacabernet center aromas noteworthy invites berries vineyards tuned longer blackberry mint dangerously spread fresher ground then spainsauvignon leader grosso (for factor distinct compliment pleasantly pismo ending lush edge re keuka culminate size concord represents underrated fresh tons bears chance brisk fruity save crimson slathered hearty sicilian foxen -length many boy .m italycoda suspended shiitake extremely date



In [32]:
info = text_generator.generate(
    "wine review : Germany", max_tokens=80, temperature=3.0
)


generated text:
wine review : Germany & fringe bud rise larry virginiared island torrontés packed onto tropical free alsacesparkling sight savory loads savor broaden chance know hits faintly sweet creating portions wrap pale nuance again earthy less named arugula valleychenin including grape -orange saint consider bulk of enduring presence 61 excellent deftness countered accompanied 500 away easygoing -rich interest st somewhere introduced lived taking south mirabelle turn castel sunset dog nectarine crushed berries erath first predominate shyly matched pushing maison (15 now–2019



In [33]:
[188]

[188]

In [34]:
[188]

[188]